# Berlin SA1 — Rebuild & SAVE the real dataset (for Phase 4)

**Goal:** Prepare a persistent, reusable **SA1 cache** for U-Net — **12 aligned spatial predictor TIFFs + 6 rainfall target TIFFs**, directly from Zenodo record `7221058`. **No RF training** and **no GPU required**.

**Important:** Previous Phase 3 downloaded/processed predictor TIFFs into `/kaggle/working` but kept targets in `/kaggle/tmp`, which disappears when the session ends. This notebook copies **both** predictor and target TIFFs into `/kaggle/working` and creates an inventory for reuse. The original huge rasters are streamed and temporary.

**Kaggle settings:** Internet **ON**, Accelerator **NONE**. Run cells from top to bottom in order. If a download fails, do **not** restart the session: rerun the failed cell, which skips completed aligned predictors.

**Saving is a SEPARATE action:** Once all 18 files exist, use **Save Version → Quick Save**, enable saving output files in **Advanced Settings** (if the option is displayed), and check the saved version's **Output** tab. If the output was not saved, do not close the session; first create a Kaggle Dataset from the files / notebook output. **Do not use Save & Run All** just to save: that starts a new full run and may download everything again.

**Scientific qualification:** Source paper used SA1/SA2 and other split procedures. This preparation is for a *technical Berlin SA1 real-data U-Net pilot*, not a reproduction of published benchmarks or a validation for Ho Chi Minh City. Some original rasters lack CRS metadata and provisional alignment must be verified before a scientific report.

## A — Set up Kaggle working and scratch folders

Data to **keep**: `/kaggle/working/berlin_sa1_aligned_predictors/*.tif` and `/kaggle/working/berlin_sa1_targets/*.tif`.
Temporary large source GeoTIFFs: `/kaggle/tmp/berlin_phase3/...` (not persistent).

In [1]:
import os, sys, subprocess, importlib.util, time, json, shutil, zlib, struct
from pathlib import Path
from collections import defaultdict

import numpy as np
RECORD='7221058'
OUT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/tmp/berlin_phase3_output')
TEMP=Path('/kaggle/tmp/berlin_phase3') if Path('/kaggle').is_dir() else Path('/tmp/berlin_phase3')
OUT.mkdir(parents=True,exist_ok=True)
TEMP.mkdir(parents=True,exist_ok=True)
CROPS=OUT/'berlin_sa1_aligned_predictors'
CROPS.mkdir(exist_ok=True)
SEED=42
TRAIN_RAIN=[40,60,80,120]
VAL_RAIN=90
TEST_RAIN=100 # Reserved paper testing event; DO NOT tune hyperparameters on these results.
N_TILES_AXIS=6
N_PIXELS_PER_TILE_EVENT=700
RF_TREES=10
print('Python:',sys.version.split()[0]); print('Output:',OUT); print('Scratch:',TEMP)
print('Scratch free GiB:',round(shutil.disk_usage(TEMP).free/2**30,2))
print('Working free GiB:',round(shutil.disk_usage(OUT).free/2**30,2))


Python: 3.13.15
Output: /kaggle/working
Scratch: /kaggle/tmp/berlin_phase3
Scratch free GiB: 1048.79
Working free GiB: 19.5


## B — Install dependencies

In [2]:
packages={'remotezip':'remotezip','rasterio':'rasterio','scikit-learn':'sklearn',
          'pandas':'pandas','matplotlib':'matplotlib','requests':'requests',
          'inflate64':'inflate64','zstandard':'zstandard','joblib':'joblib'}
missing=[pkg for pkg,mod in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import requests, rasterio, pandas as pd, joblib, sklearn
from remotezip import RemoteZip
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
print('rasterio:',rasterio.__version__, 'scikit-learn:',sklearn.__version__)


Installing: ['remotezip', 'inflate64']
rasterio: 1.5.1 scikit-learn: 1.6.1


## C — List the required input features and rainfall events

In [3]:
FEATURES=[
    'CN.tif', 'Cos_Aspect.tif', 'Curvature.tif',
    'DEM_berlin_boundary.tif','DEM_depth_1m_clip.tif', 'DEM_L.tif',
    'flow_acc_comb_nodata_clipped.tif', 'roughness_final_berlin.tif',
    'Sin_Aspect.tif', 'Slope_degree.tif', 'TPI_nan.tif',
    'TWI_Low_pass_filter_berlin_boundary.tif'
]
RAIN_SCENARIOS=sorted(set(TRAIN_RAIN+[VAL_RAIN,TEST_RAIN]))
ZIP_METHOD={0:'Stored',8:'Deflate',9:'Deflate64',12:'BZIP2',14:'ZIP LZMA',20:'Zstd',93:'Zstd'}
def zip_url(archive):
    return f'https://zenodo.org/records/{RECORD}/files/{archive}?download=1'
print('Feature TIFFs:',len(FEATURES),'rainfall targets:',RAIN_SCENARIOS)


Feature TIFFs: 12 rainfall targets: [40, 60, 80, 90, 100, 120]


## D — Remote ZIP extractor with verified range requests, retries and CRC

ZIP *members* are fetched without downloading the whole ZIP. Retries are built in for transient errors (503, 429, etc). Failed incomplete member files are cleaned up; successfully cropped predictor files are retained.

In [6]:
import inflate64
import zstandard as zstd
RANGE_CHUNK=16*2**20

class _TemporaryHTTPError(Exception):
    def __init__(self, status, retry_after=None):
        self.status = status
        self.retry_after = retry_after
        super().__init__(f"HTTP {status}")


def fetch_range(url, start, length, session):
    """Download an exact ZIP byte range, retrying transient Zenodo/network errors."""
    if length == 0:
        return b''
    end = start + length - 1
    headers = {
        'Range': f'bytes={start}-{end}',
        'Accept-Encoding': 'identity'
    }
    max_attempts = 8
    temporary_statuses = {429, 500, 502, 503, 504}
    for attempt in range(max_attempts):
        try:
            with session.get(url, headers=headers, stream=True,
                             timeout=(30, 180)) as response:
                if response.status_code in temporary_statuses:
                    raise _TemporaryHTTPError(
                        response.status_code, response.headers.get('Retry-After'))
                if response.status_code != 206:
                    raise RuntimeError(
                        f'HTTP Range unavailable: HTTP {response.status_code}. '
                        'STOP to avoid downloading an entire ZIP.')
                content_range = response.headers.get('Content-Range', '')
                if not content_range.startswith(f'bytes {start}-{end}/'):
                    raise RuntimeError('Unexpected Content-Range: ' + content_range)
                parts = bytearray()
                for block in response.iter_content(chunk_size=2**20):
                    if not block:
                        continue
                    parts.extend(block)
                    if len(parts) > length:
                        raise RuntimeError('Unexpected excess HTTP bytes')
            if len(parts) != length:
                raise IOError(f'Incomplete Range: {len(parts)} / {length}')
            return bytes(parts)
        except (_TemporaryHTTPError, requests.exceptions.RequestException, IOError) as exc:
            if attempt == max_attempts - 1:
                raise RuntimeError(
                    f'HTTP Range failed after {max_attempts} attempts '
                    f'(last error: {exc}). Saved aligned TIFFs are unaffected. '
                    'Retry this cell later if Zenodo remains unavailable.') from exc
            delay = min(60, 5 * (2 ** attempt))
            if isinstance(exc, _TemporaryHTTPError) and exc.retry_after:
                try:
                    delay = max(delay, min(180, float(exc.retry_after)))
                except (ValueError, TypeError):
                    pass
            print(f'  Temporary download problem: {exc}; '
                  f'retry {attempt+2}/{max_attempts} in {delay:.0f}s')
            time.sleep(delay)

def payload_offset(url,info,session):
    header=fetch_range(url,info.header_offset,30,session)
    (sig,ver,flags,method,mtime,mdate,crc,comp32,raw32,nlen,elen)=struct.unpack('<IHHHHHIIIHH',header)
    if sig!=0x04034b50 or method!=info.compress_type or (flags&1):
        raise RuntimeError('Unsupported encrypted or inconsistent ZIP member')
    return info.header_offset+30+nlen+elen

def extract_one(archive,name):
    filename=TEMP/archive.replace('.zip','')/name
    filename.parent.mkdir(parents=True,exist_ok=True)
    url=zip_url(archive)
    with RemoteZip(url,timeout=180) as remote:
        info=remote.getinfo(name)
        if filename.is_file() and filename.stat().st_size==info.file_size:
            print('Scratch cache:',name)
            return filename
        if shutil.disk_usage(TEMP).free < info.file_size+2*2**30:
            raise RuntimeError(f'Insufficient scratch disk for {name}: need at least '
                               f'{(info.file_size+2*2**30)/2**30:.1f} GiB free')
        tmp=filename.with_suffix(filename.suffix+'.part')
        if tmp.exists():tmp.unlink()
        crc=0;n=0;t=time.time()
        print('EXTRACT',name,'compressed GB',round(info.compress_size/1e9,2),
              'uncompressed GB',round(info.file_size/1e9,2))
        try:
            if info.compress_type in (0,8,12,14):
                with remote.open(info) as src,tmp.open('wb') as out:
                    while True:
                        data=src.read(2**20)
                        if not data:break
                        out.write(data); n+=len(data);crc=zlib.crc32(data,crc)
            elif info.compress_type in (9,20,93):
                if info.compress_type==9:
                    dec=inflate64.Inflater();decode=dec.inflate;flush=lambda:b''
                else:
                    dec=zstd.ZstdDecompressor().decompressobj();decode=dec.decompress;flush=dec.flush
                with requests.Session() as session:
                    offset=payload_offset(url,info,session)
                    with tmp.open('wb') as out:
                        for pos in range(0,info.compress_size,RANGE_CHUNK):
                            size=min(RANGE_CHUNK,info.compress_size-pos)
                            data=decode(fetch_range(url,offset+pos,size,session))
                            if data:out.write(data);n+=len(data);crc=zlib.crc32(data,crc)
                            if (pos//RANGE_CHUNK)%16==0:
                                print('  Compressed downloaded',round(min(pos+size,info.compress_size)/1e9,2),
                                      '/',round(info.compress_size/1e9,2),'GB')
                        tail=flush()
                        if tail:out.write(tail);n+=len(tail);crc=zlib.crc32(tail,crc)
            else:raise NotImplementedError('ZIP compression method: '+str(info.compress_type))
            if n!=info.file_size or (crc&0xffffffff)!=info.CRC:
                raise IOError(f'ZIP size/CRC mismatch for {name}: {n} bytes')
            tmp.replace(filename)
            print('CRC OK:',name,'seconds',round(time.time()-t,1))
        finally:
            if tmp.exists():tmp.unlink()
    return filename


## E — Download, align and save 12 predictors (expensive)

First extracts SA1/WD_60mm.tif to define the reference grid, then extracts one large predictor TIFF at a time, resamples it to SA1, saves only the aligned `float32` raster in **working**, and removes each large temporary source raster. It verifies CRS, size, and valid pixels.

This step can involve about 20 GB compressed HTTP traffic; it may be interrupted by Zenodo. If interrupted, rerun this cell in the **same session**: already completed output TIFFs are skipped.

In [7]:
# Section 4 (fixed): resume from existing aligned crops and verify missing-CRS rasters.
# A missing CRS is NOT proof that the file uses EPSG:25833.
# Only assume GRID['crs'] if the raster already has plausible Berlin UTM coordinates,
# suitable pixel size, and covers at least 95% of the SA1 target bounding box.
from rasterio.transform import array_bounds

REFERENCE=extract_one('Water_depth_TELEMAC_2D.zip','SA1/WD_60mm.tif')
with rasterio.open(REFERENCE) as ds:
    GRID=dict(crs=ds.crs,transform=ds.transform,width=ds.width,height=ds.height)
    print('SA1 reference:',GRID['crs'],GRID['width'],GRID['height'],ds.res,ds.bounds)
    assert GRID['crs'] is not None

CATEGORICAL={'CN.tif','roughness_final_berlin.tif'}

def checked_source_crs(src, name):
    if src.crs is not None:
        return src.crs, False
    # Compare numerical coordinates, NOT geography: this tests whether assuming
    # the reference CRS is plausible. It does not prove the missing CRS.
    rleft,rbottom,rright,rtop=array_bounds(GRID['height'],GRID['width'],GRID['transform'])
    b=src.bounds
    overlap_w=max(0.0,min(b.right,rright)-max(b.left,rleft))
    overlap_h=max(0.0,min(b.top,rtop)-max(b.bottom,rbottom))
    overlap=overlap_w*overlap_h/((rright-rleft)*(rtop-rbottom))
    sx,sy=map(abs,src.res)
    plausible_res=(0.25<=sx<=10 and 0.25<=sy<=10)
    plausible_xy=(100_000<b.left<900_000 and 5_000_000<b.bottom<7_000_000)
    print(f'WARNING: {name} has NO CRS. Bounds={b}, resolution={src.res}, '
          f'coverage_if_assumed={overlap:.2%}')
    if not (overlap>=.95 and plausible_res and plausible_xy):
        raise RuntimeError(
            f'Missing CRS: {name}. Cannot safely assume {GRID["crs"]}. '
            'Inspect raw raster metadata before continuing.')
    print(f'PROVISIONAL ASSUMPTION: {name} uses {GRID["crs"]} '
          '(recorded in output metadata; must be verified before paper reproduction).')
    return GRID['crs'], True

report=[]
for fi,feature in enumerate(FEATURES,1):
    output=CROPS/(Path(feature).stem+'_SA1.tif')
    if output.is_file():
        with rasterio.open(output) as cached:
            valid_cache=(cached.width==GRID['width'] and cached.height==GRID['height']
                         and cached.crs==GRID['crs']
                         and cached.transform.almost_equals(GRID['transform']))
        if valid_cache:
            print(f'[{fi}/12] KEEP cached {feature} (no download)')
            report.append(dict(feature=feature,status='cached',file=str(output)))
            pd.DataFrame(report).to_csv(OUT/'berlin_phase3_crop_progress.csv',index=False)
            continue

    raw=extract_one('Predictive_features.zip',feature)
    succeeded=False
    try:
        with rasterio.open(raw) as src:
            source_crs,assumed=checked_source_crs(src,feature)
            mode=Resampling.nearest if feature in CATEGORICAL else Resampling.bilinear
            with WarpedVRT(src,src_crs=source_crs,crs=GRID['crs'],
                           transform=GRID['transform'],width=GRID['width'],
                           height=GRID['height'],resampling=mode,
                           dtype='float32',nodata=np.nan) as vrt:
                profile=dict(driver='GTiff',width=GRID['width'],height=GRID['height'],count=1,
                             dtype='float32',crs=GRID['crs'],transform=GRID['transform'],
                             nodata=np.nan,tiled=True,blockxsize=256,blockysize=256,
                             compress='DEFLATE',predictor=3)
                partial=output.with_suffix('.partial.tif')
                if partial.exists():partial.unlink()
                count=0;total=0
                with rasterio.open(partial,'w',**profile) as dst:
                    if assumed:
                        dst.update_tags(source_crs_assumed=str(source_crs),
                                        source_crs_original='missing',
                                        assumption='SA1 bounds overlap and pixel size verified')
                    for _,win in dst.block_windows(1):
                        block=vrt.read(1,window=win,out_dtype='float32',masked=True)
                        arr=np.asarray(block.filled(np.nan),dtype='float32')
                        count+=int(np.isfinite(arr).sum());total+=arr.size
                        dst.write(arr,1,window=win)
                if count<1000:
                    partial.unlink(missing_ok=True)
                    raise RuntimeError(f'Almost no valid pixels in aligned feature {feature}; inspect CRS/NoData')
                partial.replace(output)
                succeeded=True
                print(f'[{fi}/12] SAVED {feature}: valid={count}/{total} ({count/total:.1%})')
                report.append(dict(feature=feature,status='extracted',
                                   assumed_crs=assumed,valid_fraction=count/total,file=str(output)))
    finally:
        if succeeded:
            raw.unlink(missing_ok=True)
        else:
            print('Kept source raster for debugging / retry:',raw)
    pd.DataFrame(report).to_csv(OUT/'berlin_phase3_crop_progress.csv',index=False)
print('DONE. Number of aligned predictors:',len(list(CROPS.glob('*.tif'))))


Scratch cache: SA1/WD_60mm.tif
SA1 reference: EPSG:25833 3253 3577 (1.0000298493698168, 1.0001388873357138) BoundingBox(left=384570.6118, bottom=5823429.9831, right=387823.7089, top=5827007.4799)
[1/12] KEEP cached CN.tif (no download)
EXTRACT Cos_Aspect.tif compressed GB 2.94 uncompressed GB 6.93
  Compressed downloaded 0.02 / 2.94 GB
  Compressed downloaded 0.29 / 2.94 GB
  Compressed downloaded 0.55 / 2.94 GB
  Compressed downloaded 0.82 / 2.94 GB
  Compressed downloaded 1.09 / 2.94 GB
  Compressed downloaded 1.36 / 2.94 GB
  Compressed downloaded 1.63 / 2.94 GB
  Compressed downloaded 1.9 / 2.94 GB
  Compressed downloaded 2.16 / 2.94 GB
  Compressed downloaded 2.43 / 2.94 GB
  Compressed downloaded 2.7 / 2.94 GB
CRC OK: Cos_Aspect.tif seconds 1016.1
[2/12] SAVED Cos_Aspect.tif: valid=11635981/11635981 (100.0%)
EXTRACT Curvature.tif compressed GB 2.51 uncompressed GB 6.92
  Compressed downloaded 0.02 / 2.51 GB
  Compressed downloaded 0.29 / 2.51 GB
  Compressed downloaded 0.55 / 2.5

## F — Fetch six true depth targets and COPY them to persistent output

Rainfall scenarios: **40, 60, 80, 90, 100, 120 mm**. The extractor uses scratch space; this cell COPIES successful targets into `/kaggle/working/berlin_sa1_targets` so the next notebook can read them. No interpolation or target redefinition is performed here — Phase 4 uses WarpedVRT to align targets to the SA1 grid on read.

In [8]:
TARGET_DIR=OUT/'berlin_sa1_targets'
TARGET_DIR.mkdir(parents=True,exist_ok=True)
PERSISTENT_TARGETS={}
for rain in RAIN_SCENARIOS:
    name=f'WD_{rain}mm.tif'
    dest=TARGET_DIR/name
    if dest.is_file():
        try:
            with rasterio.open(dest) as ds:
                valid_file=(ds.count>=1 and ds.crs is not None and ds.width>0 and ds.height>0)
            if valid_file:
                print(f'KEEP saved target: {name} ({dest.stat().st_size/2**20:.1f} MiB)')
                PERSISTENT_TARGETS[rain]=dest
                continue
        except Exception as e:
            print('Invalid cached target, refetching:',name,str(e))
    raw=extract_one('Water_depth_TELEMAC_2D.zip',f'SA1/{name}')
    with rasterio.open(raw) as ds:
        if ds.crs is None:
            raise RuntimeError(f'Missing CRS in target {name}')
        if ds.count<1 or ds.width<256 or ds.height<256:
            raise RuntimeError(f'Invalid target dimensions: {name}')
        print(f'Target {name}:',ds.width,ds.height,ds.crs,ds.res)
    partial=dest.with_suffix('.partial.tif')
    shutil.copy2(raw,partial)
    if partial.stat().st_size != raw.stat().st_size:
        raise RuntimeError(f'Incomplete copy of target: {name}')
    partial.replace(dest)
    PERSISTENT_TARGETS[rain]=dest
    print(f'SAVED target {name} → {dest} ({dest.stat().st_size/2**20:.1f} MiB)')
    # Delete scratch target now that we have a verified copy in working.
    if raw.resolve()!=dest.resolve():
        raw.unlink(missing_ok=True)
assert len(PERSISTENT_TARGETS)==6
print('DONE: targets saved in /kaggle/working:',len(PERSISTENT_TARGETS))


EXTRACT SA1/WD_40mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_40mm.tif seconds 32.7
Target WD_40mm.tif: 3253 3577 EPSG:25833 (1.0000298493698168, 1.0001388873357138)
SAVED target WD_40mm.tif → /kaggle/working/berlin_sa1_targets/WD_40mm.tif (88.9 MiB)
Scratch cache: SA1/WD_60mm.tif
Target WD_60mm.tif: 3253 3577 EPSG:25833 (1.0000298493698168, 1.0001388873357138)
SAVED target WD_60mm.tif → /kaggle/working/berlin_sa1_targets/WD_60mm.tif (88.9 MiB)
EXTRACT SA1/WD_80mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_80mm.tif seconds 21.7
Target WD_80mm.tif: 3253 3577 EPSG:25833 (1.0000298493698168, 1.0001388873357138)
SAVED target WD_80mm.tif → /kaggle/working/berlin_sa1_targets/WD_80mm.tif (88.9 MiB)
EXTRACT SA1/WD_90mm.tif compressed GB 0.08 uncompressed GB 0.09
CRC OK: SA1/WD_90mm.tif seconds 13.9
Target WD_90mm.tif: 3253 3577 EPSG:25833 (1.0000298493698168, 1.0001388873357138)
SAVED target WD_90mm.tif → /kaggle/working/berlin_sa1_targets/WD_90mm.tif (88.9 MiB

## G — **MANDATORY final verification and cache manifest**

Make sure there are exactly 12 predictors and 6 targets before trying Phase 4. The output lives in **working**; each file needs to be saved into a Kaggle Notebook version (or Kaggle Dataset) separately from this running session.

In [9]:
from rasterio.warp import transform_bounds
from rasterio.coords import disjoint_bounds
assert Path(CROPS).is_dir() and TARGET_DIR.is_dir()
feature_paths=[CROPS/(Path(f).stem+'_SA1.tif') for f in FEATURES]
target_paths=[TARGET_DIR/f'WD_{r}mm.tif' for r in RAIN_SCENARIOS]
for p in feature_paths+target_paths:
    assert p.is_file() and p.stat().st_size>0, f'MISSING or empty: {p}'
    with rasterio.open(p) as ds:
        assert ds.width>0 and ds.height>0 and ds.count>=1, f'Invalid raster: {p}'
        assert ds.crs is not None, f'Missing CRS: {p}'
        if p in feature_paths:
            assert ds.width==GRID['width'] and ds.height==GRID['height'],f'Shape mismatch: {p}'
            assert ds.crs==GRID['crs'] and ds.transform.almost_equals(GRID['transform']),f'Grid mismatch: {p}'
        else:
            b=transform_bounds(ds.crs,GRID['crs'],*ds.bounds,densify_pts=21)
            rb=rasterio.transform.array_bounds(GRID['height'],GRID['width'],GRID['transform'])
            assert not disjoint_bounds(b,rb),f'No overlap with SA1: {p}'
        print('OK',p.name,round(p.stat().st_size/2**20,1),'MiB')
print('PREDICTORS:',len(feature_paths),'/12; TARGETS:',len(target_paths),'/6')
print('TOTAL SAVED OUTPUT SIZE GiB:',round(sum(p.stat().st_size for p in feature_paths+target_paths)/2**30,3))
assumed_files=[]
for p in feature_paths:
    with rasterio.open(p) as ds:
        if ds.tags().get('source_crs_original')=='missing': assumed_files.append(p.name)
manifest={
 'status':'SA1_PREPARED_DATA_FOR_BERLIN_UNET_PILOT_NOT_PAPER_REPRODUCTION',
 'source_record':'7221058','paper_doi':'10.5194/nhess-23-809-2023',
 'feature_order':FEATURES,
 'rainfall_mm':RAIN_SCENARIOS,'grid_crs':str(GRID['crs']),
 'grid_width':int(GRID['width']),'grid_height':int(GRID['height']),
 'grid_transform':list(GRID['transform'])[:6],
 'predictor_files':[str(p.relative_to(OUT)) for p in feature_paths],
 'target_files':[str(p.relative_to(OUT)) for p in target_paths],
 'output_size_gib':round(sum(p.stat().st_size for p in feature_paths+target_paths)/2**30,3),
 'crs_assumptions':list(assumed_files),
 'cautions':['provisional CRS assumptions for some source rasters','rainfall scenario train/test protocol is a pilot','units of depth must be verified at source']
}
(OUT/'berlin_sa1_cache_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print('\nDATASET READY. SAVE THE OUTPUT FILES NOW before ending this session.')
print('Cache manifest:',OUT/'berlin_sa1_cache_manifest.json')


OK CN_SA1.tif 0.3 MiB
OK Cos_Aspect_SA1.tif 39.6 MiB
OK Curvature_SA1.tif 39.9 MiB
OK DEM_berlin_boundary_SA1.tif 23.0 MiB
OK DEM_depth_1m_clip_SA1.tif 24.8 MiB
OK DEM_L_SA1.tif 39.5 MiB
OK flow_acc_comb_nodata_clipped_SA1.tif 34.0 MiB
OK roughness_final_berlin_SA1.tif 1.2 MiB
OK Sin_Aspect_SA1.tif 39.6 MiB
OK Slope_degree_SA1.tif 35.5 MiB
OK TPI_nan_SA1.tif 39.9 MiB
OK TWI_Low_pass_filter_berlin_boundary_SA1.tif 33.1 MiB
OK WD_40mm.tif 88.9 MiB
OK WD_60mm.tif 88.9 MiB
OK WD_80mm.tif 88.9 MiB
OK WD_90mm.tif 88.9 MiB
OK WD_100mm.tif 88.9 MiB
OK WD_120mm.tif 88.9 MiB
PREDICTORS: 12 /12; TARGETS: 6 /6
TOTAL SAVED OUTPUT SIZE GiB: 0.863

DATASET READY. SAVE THE OUTPUT FILES NOW before ending this session.
Cache manifest: /kaggle/working/berlin_sa1_cache_manifest.json


## H — SAVE FIRST, then Phase 4

1. Click **Save Version → Quick Save**. In **Advanced Settings**, turn on **Save output files** if it exists. **Do not** click Save & Run All (that runs the 20 GB download from zero in a fresh session).
2. Open the saved version, go to **Output**. Check that both subfolders exist: `berlin_sa1_aligned_predictors` (12 TIFFs) and `berlin_sa1_targets` (6 TIFFs). If not, the data is NOT yet durable; do not close your current session.
3. On the saved notebook Output page, use **Create Dataset** (if available) to persist it as a Kaggle Dataset. Alternatively, in a new Phase 4 notebook use **Add Input** to attach the saved notebook output. Both methods make the prepared data available under `/kaggle/input/<dataset-slug>/...`.
4. Open the existing **Berlin Phase 4 — Real-data U-Net** notebook; **enable GPU T4 BEFORE running any cells**, then **Add Input** with the saved dataset/notebook output. Run from the first cell. It should print 12 predictors and six targets rather than raising FileNotFoundError.
5. Run 1 epoch smoke pilot and save: `berlin_phase4_unet_best.keras`, `berlin_phase4_metrics.csv`, `berlin_phase4_summary.json`, `berlin_phase4_real_prediction.png`.

If any step fails, show the actual traceback. Avoid re-downloading while a previously saved version or cache exists.

In [10]:
from pathlib import Path

p = Path("/kaggle/working/berlin_sa1_aligned_predictors")
t = Path("/kaggle/working/berlin_sa1_targets")

print("Predictors:", len(list(p.glob("*.tif"))))
print("Targets:", len(list(t.glob("*.tif"))))

Predictors: 12
Targets: 6


In [11]:

from pathlib import Path
import shutil
import kagglehub

# 1. Locate the prepared Berlin data
base = Path("/kaggle/working")
predictors = base / "berlin_sa1_aligned_predictors"
targets = base / "berlin_sa1_targets"

n_predictors = len(list(predictors.glob("*.tif")))
n_targets = len(list(targets.glob("*.tif")))

print("Predictors:", n_predictors)
print("Targets:", n_targets)

assert n_predictors == 12, "Missing spatial predictors"
assert n_targets == 6, "Missing water-depth targets"

# 2. Prepare an isolated upload folder
upload_dir = base / "berlin_sa1_upload"
upload_dir.mkdir(exist_ok=True)

shutil.copytree(
    predictors,
    upload_dir / predictors.name,
    dirs_exist_ok=True
)

shutil.copytree(
    targets,
    upload_dir / targets.name,
    dirs_exist_ok=True
)

manifest = base / "berlin_sa1_cache_manifest.json"
if manifest.exists():
    shutil.copy2(manifest, upload_dir / manifest.name)

# 3. Upload to Kaggle Datasets
handle = "longltrnhong/berlin-sa1-prepared"

kagglehub.dataset_upload(handle, str(upload_dir))

print("\nUPLOAD FINISHED")
print("Dataset:", f"https://www.kaggle.com/datasets/{handle}")


Predictors: 12
Targets: 6
Uploading Dataset https://kaggle.com/datasets/longltrnhong/berlin-sa1-prepared ...
Starting upload for file /kaggle/working/berlin_sa1_upload/berlin_sa1_cache_manifest.json

Uploading: 100%|██████████| 2.06k/2.06k [00:00<00:00, 4.94kB/s]

Upload successful: /kaggle/working/berlin_sa1_upload/berlin_sa1_cache_manifest.json (2KB)
Starting upload for file /kaggle/working/berlin_sa1_upload/berlin_sa1_aligned_predictors/Slope_degree_SA1.tif



Uploading: 100%|██████████| 37.3M/37.3M [00:00<00:00, 38.7MB/s]

Upload successful: /kaggle/working/berlin_sa1_upload/berlin_sa1_aligned_predictors/Slope_degree_SA1.tif (36MB)
Starting upload for file /kaggle/working/berlin_sa1_upload/berlin_sa1_aligned_predictors/flow_acc_comb_nodata_clipped_SA1.tif



Uploading: 100%|██████████| 35.6M/35.6M [00:00<00:00, 40.8MB/s]

... (30 outputs left out)

Upload successful: /kaggle/working/berlin_sa1_upload/berlin_sa1_targets/WD_40mm.tif (89MB)
Starting upload for file /kaggle/working/berlin_sa1_upload/berlin_sa1_targets/WD_120mm.tif



Uploading: 100%|██████████| 93.2M/93.2M [00:05<00:00, 18.2MB/s]

Upload successful: /kaggle/working/berlin_sa1_upload/berlin_sa1_targets/WD_120mm.tif (89MB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/longltrnhong/berlin-sa1-prepared

UPLOAD FINISHED
Dataset: https://www.kaggle.com/datasets/longltrnhong/berlin-sa1-prepared
